# 🌫️ Aab-O-Hawa AQI Predictor — Project Report

> **Author:** Muhammad Taqee  
> **Project:** Aab-O-Hawa — Karachi Air Quality Intelligence System  
> **Stack:** Python · Open-Meteo API · MongoDB Atlas · GitHub Actions · Streamlit  
> **Period:** Feb 2026 – Jun 2026

---

This report documents the major technical challenges encountered while building an end-to-end AQI prediction and monitoring system for Karachi, Pakistan — and how each was resolved.

## 1. Project Overview

**Aab-O-Hawa** (آب و ہوا) is a Farsi/Urdu phrase meaning *air and atmosphere*. The project is a fully automated, serverless AQI prediction system built for Karachi.

### Goals
- Monitor real-time air quality in Karachi
- Forecast AQI for the next 3 days
- Issue health alerts for hazardous conditions
- Automate the entire data → model → dashboard pipeline

### Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    AABO-O-HAWA ARCHITECTURE                  │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Open-Meteo API          GitHub Actions (hourly cron)        │
│  ├── Air Quality API  ──► Feature Pipeline  ──► MongoDB Atlas│
│  └── Archive/Forecast     (feature_pipeline.py)  │          │
│                                                  │          │
│                           GitHub Actions (daily) │          │
│                           Training Pipeline  ◄───┘          │
│                           (train_aqi_model.py)              │
│                                    │                         │
│                                    ▼                         │
│                           MongoDB Model Registry             │
│                           ├── model_registry                 │
│                           ├── aqi_forecasts                  │
│                           └── aqi_forecasts_hourly           │
│                                    │                         │
│                                    ▼                         │
│                           Streamlit Dashboard                │
│                           (app.py → Streamlit Cloud)         │
└─────────────────────────────────────────────────────────────┘
```

### Tech Stack

| Layer | Technology |
|-------|------------|
| Data Source | Open-Meteo Air Quality + Archive API |
| Feature Store | MongoDB Atlas (free tier) |
| ML Models | Random Forest, Ridge, XGBoost, Gradient Boosting |
| Explainability | SHAP (SHapley Additive exPlanations) |
| Automation | GitHub Actions (hourly + daily cron) |
| Dashboard | Streamlit + Plotly |
| Deployment | Streamlit Cloud |

---
## Problems & Solutions

> The following sections document each major problem encountered, its root cause, and the solution applied.

---
## Problem 1 — Hopsworks Migration to MongoDB

### What happened
The project originally used **Hopsworks** as the feature store. After setting it up and running the pipeline, several blockers emerged:
- Confluent Kafka and PyArrow dependencies were missing and caused repeated GitHub Actions failures
- The free tier had compute job limits that throttled hourly inserts
- The `datetime` column caused conflicts since Hopsworks uses it as a reserved keyword
- Insert jobs had significant delays (`wait_for_job=True` would time out)

### Root Cause
Hopsworks is a heavy enterprise platform that requires multiple optional dependencies (`pyarrow`, `confluent-kafka`) that are not auto-installed. Its free tier compute limits also make frequent small inserts impractical.

### Solution
Migrated the entire feature store to **MongoDB Atlas** (free M0 tier) which:
- Has no compute job limits
- Supports instant upserts via `bulk_write`
- Stores raw documents without schema conflicts
- Requires only `pymongo` and `dnspython`

In [ ]:
# Hopsworks approach (problematic)
# fg.insert(df, write_options={"wait_for_job": True})  # slow, compute-limited

# MongoDB approach (solution)
from pymongo import MongoClient, UpdateOne

client     = MongoClient("your_mongo_uri")
collection = client["karachi_aqi"]["hourly_features"]

records = df.to_dict("records")
ops = [UpdateOne({"time": r["time"]}, {"$set": r}, upsert=True) for r in records]
result = collection.bulk_write(ops)
print(f"Upserted: {result.upserted_count} | Modified: {result.modified_count}")

---
## Problem 2 — Dependency Version Conflicts in GitHub Actions

### What happened
GitHub Actions failed repeatedly with errors like:
```
ERROR: Cannot install hopsworks==4.7.5 and pandas==3.0.3 because these package 
versions have conflicting dependencies.
hopsworks 4.7.5 depends on pandas<2.4.0
```
Later, even after switching to MongoDB:
```
ModuleNotFoundError: No module named 'pyarrow'
ModuleNotFoundError: No module named 'confluent_kafka'
ModuleNotFoundError: No module named 'xgboost'
```

### Root Cause
1. Exact version pins (`==`) in `requirements.txt` caused resolution conflicts
2. Optional dependencies were not explicitly listed
3. On Windows, packages were installed in the global Python instead of the virtual environment

### Solution
Use minimum version pins (`>=`) with upper bounds only where necessary, and explicitly list all required packages.

In [ ]:
# ❌ Before — exact pins caused conflicts
# hopsworks==4.7.5
# numpy==2.4.4
# pandas==3.0.3

# ✅ After — loose pins with compatibility bounds
requirements = """
pymongo>=4.0
dnspython>=2.0
pandas>=1.5,<2.4
numpy>=1.24,<2.0
requests>=2.28
python-dotenv>=1.0
scikit-learn>=1.3
xgboost>=1.7
joblib>=1.3
pyarrow>=12.0
"""
print(requirements)

---
## Problem 3 — Feature Pipeline Uploading Same Data Repeatedly

### What happened
After 38 GitHub Actions runs, MongoDB still only had 2,160 rows — the same count as the initial 90-day backfill. Every hourly run was uploading the same rows instead of new ones.

### Root Cause
The date range was hardcoded:
```python
END_DATE   = date.today() - timedelta(days=1)
START_DATE = END_DATE - timedelta(days=1)   # always same 2 days
```
MongoDB's `upsert=True` silently overwrote the same rows each time without adding new ones.

### Solution
Query MongoDB for the last uploaded timestamp and use it as the start date, then filter results to only upload genuinely new rows.

In [ ]:
import pandas as pd
from datetime import date, timedelta

# ❌ Before — always same date range
# END_DATE   = date.today() - timedelta(days=1)
# START_DATE = END_DATE - timedelta(days=1)

# ✅ After — smart date range based on last uploaded record
END_DATE = date.today() - timedelta(days=1)

last_doc = collection.find_one(sort=[("time", -1)])
if last_doc:
    last_date  = pd.Timestamp(last_doc["time"]).date()
    START_DATE = last_date - timedelta(days=2)   # small overlap for lag features
    print(f"Last record: {last_date}")
else:
    START_DATE = END_DATE - timedelta(days=90)   # first run backfill

# filter only new rows before uploading
if last_doc:
    last_uploaded = pd.Timestamp(last_doc["time"]).tz_localize(None)
    df = df[df["time"] > last_uploaded]
    print(f"New rows to upload: {len(df)}")

---
## Problem 4 — Empty DataFrame After `dropna()` in Feature Pipeline

### What happened
```
pymongo.errors.InvalidOperation: No operations to execute
```
The pipeline fetched data but after feature engineering and `dropna()`, the DataFrame was completely empty.

### Root Cause
Lag features like `aqi_lag_24h` and `aqi_roll_24h` require at least 24 previous rows to compute. When the pipeline fetched only 1-2 days of data, all rows became `NaN` after the shift operations — and `dropna()` removed everything.

### Solution
Two fixes applied together:
1. Fetch 30 days of context (even if only uploading the new rows)
2. Use `dropna(subset=[...])` to only drop rows where the **target** is NaN, not all columns

In [ ]:
import pandas as pd
import numpy as np

# Simulate the problem
df_small = pd.DataFrame({"us_aqi": [100, 102, 98]})  # only 3 rows
df_small["aqi_lag_24h"] = df_small["us_aqi"].shift(24)  # all NaN
df_small.dropna(inplace=True)
print(f"Rows after dropna on small df: {len(df_small)}")  # → 0

# ✅ Fix 1 — fetch 30 days back so lags have enough context
# START_DATE = last_date - timedelta(days=30)

# ✅ Fix 2 — only drop rows where TARGET is NaN, not feature NaNs
df = pd.DataFrame({"us_aqi": range(100)})  
df["aqi_lag_24h"]  = df["us_aqi"].shift(24)
df["aqi_roll_24h"] = df["us_aqi"].rolling(24).mean()
df["target"]       = df["us_aqi"].shift(-1)

# only drop where target is missing — not where features are NaN
df.dropna(subset=["target"], inplace=True)
df.fillna(0, inplace=True)  # fill remaining feature NaNs safely
print(f"Rows after targeted dropna: {len(df)}")  # → 99

---
## Problem 5 — Open-Meteo Archive API JSON Decode Error

### What happened
GitHub Actions failed with:
```
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
requests.exceptions.JSONDecodeError: Expecting value: line 1 column 1
```

### Root Cause
The Open-Meteo **archive API** (`archive-api.open-meteo.com`) has a processing delay of 2–5 days. Requesting yesterday's data returned an empty HTTP response — calling `.json()` on an empty response raised the error.

### Solution
Two fixes:
1. Probe the API dynamically to find the latest available date
2. Validate the API response before parsing

In [ ]:
import requests
from datetime import date, timedelta

LAT, LON = 24.8608, 67.0104

def get_latest_available_date():
    """Probe archive API to find most recent date with actual data."""
    for days_back in range(1, 10):
        probe_date = date.today() - timedelta(days=days_back)
        resp = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude": LAT, "longitude": LON,
                "start_date": str(probe_date), "end_date": str(probe_date),
                "hourly": ["temperature_2m"], "timezone": "Asia/Karachi",
            }, timeout=15
        )
        if resp.status_code == 200 and resp.text.strip():
            try:
                temps = resp.json()["hourly"].get("temperature_2m", [])
                if temps and any(v is not None for v in temps):
                    print(f"Latest available: {probe_date}")
                    return probe_date
            except Exception:
                continue
    return date.today() - timedelta(days=7)  # safe fallback

# Always validate before parsing
resp = requests.get("https://archive-api.open-meteo.com/v1/archive",
                    params={}, timeout=30)
if resp.status_code != 200 or not resp.text.strip():
    print(f"API error {resp.status_code} — skipping run")
else:
    data = resp.json()

---
## Problem 6 — Data Leakage Causing Artificially Perfect Model Accuracy

### What happened
Model metrics looked suspiciously perfect:
```
Ridge Regression    RMSE=0.08   R²=1.0000  ✅ Excellent
Random Forest       RMSE=0.74   R²=0.9983  ✅ Excellent
```

### Root Cause
The model was predicting `us_aqi` (current AQI) using `aqi_lag_1h` (AQI from 1 hour ago) as a feature. Since consecutive AQI readings are nearly identical, the model simply learned `prediction ≈ aqi_lag_1h` — essentially copying the previous value.

This is **data leakage**: a feature that is too similar to the target makes accuracy artificially high but the model has no real forecasting ability.

### Solution
Changed the target to **future AQI** (`shift(-24)` for next day) and removed short-horizon lags (1h, 2h, 3h) from features, keeping only `lag_24h`, `lag_48h`, `lag_72h`.

In [ ]:
import pandas as pd
import numpy as np

# ❌ Data leakage — predicting current AQI using last hour's AQI
# TARGET = "us_aqi"               # current AQI
# FEATURES includes "aqi_lag_1h" # AQI 1 hour ago — nearly identical!

# ✅ Honest forecast — predicting FUTURE AQI
df = pd.DataFrame({"us_aqi": range(200)})

# only use lags that are realistic for multi-day forecasting
for lag in [24, 48, 72]:       # ✅ 1-3 days ago
    df[f"aqi_lag_{lag}h"] = df["us_aqi"].shift(lag)

# ❌ remove these — too close to current value
# for lag in [1, 2, 3, 6, 12]:  
#     df[f"aqi_lag_{lag}h"] = df["us_aqi"].shift(lag)

# target = next day's AQI, not current
df["target"] = df["us_aqi"].shift(-24)  # ✅ what AQI will be 24h from now

df.dropna(subset=["target"], inplace=True)
print("Features used:", [c for c in df.columns if c not in ["us_aqi", "target"]])

---
## Problem 7 — SHAP Waterfall Plot TypeError

### What happened
```
TypeError: only 0-dimensional arrays can be converted to Python scalars
  base_values = float(shap_values.base_values)
```

### Root Cause
`explainer.expected_value` returned a numpy array instead of a scalar float when using `RandomForestRegressor`. SHAP's `waterfall_plot` expects a plain Python float for `base_values`.

### Solution
Safely extract the scalar value with a type check before passing it to the waterfall plot.

In [ ]:
import shap
import numpy as np

# explainer.expected_value may return array or scalar depending on SHAP version
# explainer = shap.TreeExplainer(model)

# ❌ This crashes if expected_value is an array
# base_val = explainer.expected_value

# ✅ Safe extraction — handles both array and scalar
def get_base_value(explainer):
    base_val = explainer.expected_value
    if hasattr(base_val, "__len__"):
        return float(base_val[0])   # array → take first element
    return float(base_val)          # scalar → direct conversion

# Usage example
# base_val = get_base_value(explainer)
# shap.waterfall_plot(shap.Explanation(
#     values        = shap_values[worst_idx],
#     base_values   = base_val,        # ← now always a scalar
#     data          = X_test.iloc[worst_idx].values,
#     feature_names = FEATURES
# ))
print("Safe base_value extraction defined")

---
## Problem 8 — GitHub Actions Scheduled Runs Delayed

### What happened
The workflow was scheduled to run every hour (`cron: "0 * * * *"`) but runs were delayed by 1–3 hours during peak times:
```
Run #5  5:24 PM  → Scheduled
Run #6  8:16 PM  → Scheduled  (gap: ~3 hours)
Run #7  10:48 PM → Scheduled  (gap: ~2.5 hours)
```

### Root Cause
GitHub explicitly states in their documentation that scheduled workflows may be delayed by up to several hours during periods of high load on shared runners. The free tier offers no scheduling guarantees.

### Solution
The delay was accepted as a known limitation since the data pipeline fetches the last 24 hours on every run — no hourly data is ever lost even if a run is delayed. For stricter scheduling, an external trigger service (e.g. cron-job.org) can dispatch workflow runs via the GitHub API.

In [ ]:
# GitHub Actions workflow with manual dispatch fallback
workflow_yml = """
name: Karachi AQI — Feature Pipeline

on:
  schedule:
    - cron: "0 * * * *"   # every hour (may be delayed)
  workflow_dispatch:       # ← enables manual trigger as backup

jobs:
  feature-pipeline:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements.txt
      - run: python feature_pipeline.py
        env:
          MONGO_URI: ${{ secrets.MONGO_URI }}
"""
print(workflow_yml)

---
## Problem 9 — Visibility Column Showing 0.0 km

### What happened
The Streamlit dashboard showed **Visibility: 0.0 km** for all readings. Running a quality check confirmed:
```
No non-zero visibility values found — cannot compute median
```
All 2,160+ visibility rows in MongoDB were zero.

### Root Cause
The Open-Meteo API returns `null` for visibility in most hourly records for Karachi — their dataset does not cover this variable reliably for this region. The feature pipeline's blanket `df.fillna(0)` converted all nulls to zeros, which then got stored in MongoDB.

### Solution
1. Removed the `visibility` column from all MongoDB documents using `$unset`
2. Removed `visibility` from the API fetch parameters
3. Added smarter null filling that uses median for physical variables instead of zero

In [ ]:
from pymongo import MongoClient

# ── Remove bad column from all MongoDB documents ──────────────────────────────
# client = MongoClient("your_uri")
# col    = client["karachi_aqi"]["hourly_features"]

# result = col.update_many({}, {"$unset": {"visibility": ""}})
# print(f"Removed visibility from {result.modified_count} documents")

# ── Smarter null filling in pipeline ─────────────────────────────────────────
import pandas as pd

# columns where 0 is physically impossible — use median
no_zero_cols = [
    "surface_pressure", "wind_speed_10m",
    "temperature_2m",   "relative_humidity_2m",
    "shortwave_radiation"
]

df = pd.DataFrame({
    "surface_pressure": [1013, None, 1010],
    "cloud_cover":      [None, 50, 0],
})

for col in df.select_dtypes("number").columns:
    if df[col].isnull().any():
        if col in no_zero_cols:
            df[col].fillna(df[col].median(), inplace=True)  # physical vars → median
        else:
            df[col].fillna(0, inplace=True)                 # counts/rates → 0

print(df)

---
## Problem 10 — Streamlit App Showing Stale Data on Cloud

### What happened
After deploying to Streamlit Cloud, the app showed outdated data even though MongoDB had been updated. The app only showed fresh data after manually rebooting it from the Streamlit Cloud dashboard.

### Root Cause
Two caching issues:
1. `@st.cache_resource` on `get_db()` cached the **MongoDB connection permanently** — the same stale connection was reused across all sessions
2. `@st.cache_data(ttl=1800)` meant data was cached for 30 minutes regardless of what was in MongoDB

### Solution
1. Removed `@st.cache_resource` from `get_db()` — fresh connection on every call
2. Reduced `ttl` from 1800 to 300 (5 minutes) on all data functions
3. Used `st.secrets` instead of `os.getenv` for Streamlit Cloud compatibility

In [ ]:
import streamlit as st
import os
from pymongo import MongoClient

# ❌ Before — cached connection serves stale data
# @st.cache_resource
# def get_db():
#     client = MongoClient(st.secrets["MONGO_URI"])
#     return client[st.secrets.get("MONGO_DB", "karachi_aqi")]

# @st.cache_data(ttl=1800)   # 30 minutes — too long
# def load_data(): ...


# ✅ After — fresh connection, short TTL
def get_db():   # no cache decorator
    uri     = st.secrets.get("MONGO_URI",  os.getenv("MONGO_URI"))
    db_name = st.secrets.get("MONGO_DB",   os.getenv("MONGO_DB", "karachi_aqi"))
    client  = MongoClient(uri, serverSelectionTimeoutMS=10000)
    return client[db_name]

@st.cache_data(ttl=300)   # 5 minutes — refreshes automatically
def load_current_aqi():
    return get_db()["hourly_features"].find_one(sort=[("time", -1)])

print("Fixed: get_db() has no cache, data TTL is 5 minutes")

---

## Summary — Problems 1–10

| # | Problem | Root Cause | Solution |
|---|---------|-----------|----------|
| 1 | Hopsworks migration | Compute limits + missing deps | Migrated to MongoDB Atlas |
| 2 | Dependency conflicts | Exact version pins in requirements | Loose version pins (`>=`) |
| 3 | Same data uploaded repeatedly | Hardcoded date range | Smart date range from last record |
| 4 | Empty DataFrame after dropna | Lag features need 24+ rows | Fetch 30-day context, targeted dropna |
| 5 | JSON decode error from API | Archive API has 2–5 day lag | Probe API for latest available date |
| 6 | Artificially perfect accuracy | Data leakage via `aqi_lag_1h` | Predict future AQI, remove short lags |
| 7 | SHAP waterfall TypeError | `expected_value` returned array | Safe scalar extraction with type check |
| 8 | GitHub Actions delays | GitHub free tier scheduling | Accepted + added manual dispatch |
| 9 | Visibility showing 0.0 km | API returns null for Karachi | Removed column, smarter null filling |
| 10 | Stale data on Streamlit Cloud | Permanent connection cache | Removed cache from `get_db()`, TTL=300 |

---

> **Prompt the author to continue with Problems 11–20.**